# Train each branch & full hybrid detector (Kaggle GPU)

Trains the detector's branches and evaluates using the ForenSynths cross-generator evaluation protocol:

| mode | branch | input | how it trains |
|---|---|---|---|
| `semantic_only` | CLIP ViT-L/14 semantic embedding | cached features | `train_cached.py`, minutes |
| `physics_only` | 22-d physics features | cached features | `train_cached.py`, minutes |
| `freq_only` | 64-d frequency/upsampling features | cached features | `train_cached.py`, minutes |
| `prnu_only` | PRNU sensor-noise features (optional) | cached features | `train_cached.py`, minutes |
| `stage1` | deep branch (EfficientNet on pixels) | raw images | `train.py`, hours per epoch |
| `full_hybrid` | 4-branch gated hybrid with branch dropout | pixels + cache | `train.py`, hours |

**One-time setup:** Accelerator -> GPU (T4/P100) - Internet -> On - add dataset as input - Add-ons -> Secrets -> `HF_TOKEN` (write access) toggled on.


In [10]:
# 1. Config
import os
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/Algorithm-bot/ai-vs-real-face-detector.git"
KAGGLE_DATASET_MOUNT = Path(
    "/kaggle/input/datasets/sahilsawantspit/ai-vs-real-face-detector-data"
)

# ---- Which branches to train, in order ----
# "freq_only" replaces prnu_only by default. "stage1" trains deep branch on pixels.
BRANCHES = ["semantic_only", "physics_only", "freq_only", "stage1"]

HPARAMS = {
    # cached branches: train_cached.py settings
    "semantic_only": dict(epochs=60, batch_size=128, lr=1e-3),
    "physics_only":  dict(epochs=60, batch_size=128, lr=1e-3),
    "freq_only":     dict(epochs=60, batch_size=128, lr=1e-3),
    "prnu_only":     dict(epochs=60, batch_size=128, lr=1e-3),
    # deep branch
    "stage1":        dict(epochs=4, batch_size=32,  lr=1e-4),
}

FORCE_RETRAIN = False
FAST_SCALER_FIT = True    # fit scalers from the cache instead of re-extracting images

MODELS_ROOT = Path("/kaggle/working/models_v2")
FEATURE_CACHE_DIR = Path("/kaggle/working/feature_cache_v2")
NUM_WORKERS = 3           # workers for DataLoader

HF_CHECKPOINT_REPO = "sahilSpit/checkpoints"
HF_CHECKPOINT_PRIVATE = True

unknown = [m for m in BRANCHES if m not in HPARAMS]
assert not unknown, f"No HPARAMS entry for: {unknown}"
for mode in BRANCHES:
    (MODELS_ROOT / mode).mkdir(parents=True, exist_ok=True)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

assert KAGGLE_DATASET_MOUNT.exists(), (
    f"{KAGGLE_DATASET_MOUNT} not found -- add your dataset as input first."
)
print("Dataset mounted at:", KAGGLE_DATASET_MOUNT)
print("Training branches:", BRANCHES)


Dataset mounted at: /kaggle/input/datasets/sahilsawantspit/ai-vs-real-face-detector-data
Training branches: ['semantic_only', 'physics_only', 'prnu_only', 'stage1']


In [11]:
# 2. Clone repo and install requirements
import subprocess
import shutil
from pathlib import Path

clone_dir = Path("/kaggle/working/ai-vs-real-face-detector")

if clone_dir.exists():
    shutil.rmtree(clone_dir)

# Clone hybrid-detector-eval-overhaul branch
subprocess.run([
    "git", "clone",
    "--depth", "1",
    "--branch", "hybrid-detector-eval-overhaul",
    GITHUB_REPO_URL,
    str(clone_dir)
], check=True)

requirements_files = list(clone_dir.rglob("requirements.txt"))
if not requirements_files:
    raise FileNotFoundError("requirements.txt not found in cloned repository")
requirements_file = requirements_files[0]

# Install dependencies (timm is preinstalled on Kaggle)
subprocess.run([
    "pip", "install", "-q",
    "-r", str(requirements_file)
], check=True)

project_root = requirements_file.parent
print("\nRepo ready at:", clone_dir)
print("Project root:", project_root)


Cloning into '/kaggle/working/ai-vs-real-face-detector'...


Requirements files found:
   /kaggle/working/ai-vs-real-face-detector/ai-vs-real-face-detector/requirements.txt

Repo ready at: /kaggle/working/ai-vs-real-face-detector
Project root: /kaggle/working/ai-vs-real-face-detector/ai-vs-real-face-detector
Requirements: /kaggle/working/ai-vs-real-face-detector/ai-vs-real-face-detector/requirements.txt


In [ ]:
# 2b. Run Kaggle Verification Report
# Verifies expected counts, 20-image cache-vs-live parity, DEV/REPORT splits, and throughput
import subprocess, sys

verify_cmd = [
    sys.executable, str(project_root / "kaggle_verify.py"),
    "--data-dir", str(KAGGLE_DATASET_MOUNT),
    "--feature-cache", str(FEATURE_CACHE_DIR),
]
print("Running kaggle_verify.py...")
subprocess.run(verify_cmd, check=False)


In [14]:
# 3. HF auth (used to restore/sync checkpoints and the feature cache)
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_CHECKPOINT_REPO, repo_type="model", private=HF_CHECKPOINT_PRIVATE, exist_ok=True)
print("Checkpoint repo ready:", HF_CHECKPOINT_REPO)


Checkpoint repo ready: sahilSpit/checkpoints


In [15]:
# 4a. Restore from HF, then keep syncing each branch's checkpoints + the feature cache
import threading, time
from huggingface_hub import snapshot_download

WORKING = Path("/kaggle/working")
SYNC_DIRS = {FEATURE_CACHE_DIR: "feature_cache_v2"}
for mode in BRANCHES:
    SYNC_DIRS[MODELS_ROOT / mode] = f"models_v2/{mode}"
uploaded = {}  # local path -> mtime already uploaded

# Restore whatever a previous session (yours or a friend's) pushed
try:
    snapshot_download(
        repo_id=HF_CHECKPOINT_REPO, repo_type="model", token=hf_token,
        local_dir=str(WORKING),
        allow_patterns=["feature_cache_v2/*"] + [f"models_v2/{m}/*" for m in BRANCHES],
    )
except Exception as e:
    print("Nothing to restore (fresh start):", e)
print("Feature cache files:", sorted(p.name for p in FEATURE_CACHE_DIR.glob("*.npz")))
for mode in BRANCHES:
    print(f"  {mode}:", sorted(p.name for p in (MODELS_ROOT / mode).glob("*") if p.is_file()))

# Don't re-upload what we just downloaded
for d in SYNC_DIRS:
    for f in d.rglob("*"):
        if f.is_file():
            uploaded[str(f)] = f.stat().st_mtime

def sync_once(min_age=60):
    for local_dir, repo_dir in SYNC_DIRS.items():
        for f in local_dir.rglob("*"):
            if not f.is_file():
                continue
            st = f.stat()
            if time.time() - st.st_mtime < min_age:   # probably still being written
                continue
            if uploaded.get(str(f)) == st.st_mtime:
                continue
            try:
                api.upload_file(
                    path_or_fileobj=str(f),
                    path_in_repo=f"{repo_dir}/{f.relative_to(local_dir).as_posix()}",
                    repo_id=HF_CHECKPOINT_REPO, repo_type="model",
                )
                uploaded[str(f)] = st.st_mtime
                print("synced", f.name, flush=True)
            except Exception as e:
                print("sync failed for", f.name, e, flush=True)

stop_sync = threading.Event()
def _loop():
    while not stop_sync.is_set():
        sync_once()
        stop_sync.wait(700)   # every 2 min (stage1 writes a checkpoint once per epoch)
threading.Thread(target=_loop, daemon=True).start()
print("Background sync started")

log_path = Path("/kaggle/working/train.log")

def _log_loop():
    while not stop_sync.is_set():
        if log_path.exists():
            try:
                api.upload_file(path_or_fileobj=str(log_path), path_in_repo="logs/train_branches.log",
                                repo_id=HF_CHECKPOINT_REPO, repo_type="model")
            except Exception as e:
                print("log sync failed:", e, flush=True)
        stop_sync.wait(120)   # every 2 min
threading.Thread(target=_log_loop, daemon=True).start()


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 53 files:   0%|          | 0/53 [00:00<?, ?it/s]

Feature cache files: ['test.npz', 'train.npz', 'val.npz']
  semantic_only: ['semantic_only_best.pt', 'semantic_only_history.json', 'semantic_only_last.pt']
  physics_only: ['physics_only_best.pt', 'physics_only_history.json', 'physics_only_last.pt']
  prnu_only: ['prnu_only_best.pt', 'prnu_only_history.json', 'prnu_only_last.pt']
  stage1: ['stage1_best.pt', 'stage1_history_partial.json', 'stage1_last.pt']
Background sync started


In [16]:
# 4b. Multi-branch feature cache (CLIP ViT-L/14 + Physics + Frequency)
import subprocess, sys

# Builds train, val, and test cache shards with GPU-accelerated fp16 CLIP extraction,
# frequency features, and physics features. Uses --augment-train for raw-resolution augmentations.
cache_cmd = [
    sys.executable, str(project_root / "src" / "cache_features.py"),
    "--data-dir", str(KAGGLE_DATASET_MOUNT),
    "--cache-dir", str(FEATURE_CACHE_DIR),
    "--workers", str(NUM_WORKERS),
    "--semantic-model", "vit_large_patch14_clip_224.openai",
    "--preprocess", "crop",
    "--augment-train",
    "--batch-size", "64",
]
print("Building feature cache:", " ".join(cache_cmd))
subprocess.run(cache_cmd, check=False, env={**os.environ, "PYTHONUNBUFFERED": "1"})

cache_complete = all((FEATURE_CACHE_DIR / f"{s}.npz").exists() for s in ["train", "val", "test"])
print("Cache complete status:", cache_complete)


No files have been modified since last commit. Skipping to prevent empty commit.
Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Loading cached file list for train...
[train] already cached -> /kaggle/working/feature_cache_aug/train.npz
Feature cache complete: /kaggle/working/feature_cache_aug


In [18]:
# 5. Train each branch on its own
import json, time, subprocess, sys

def run_and_log(cmd, extra_env=None):
    # Stream output live, thin out tqdm bars to one every 30s, and mirror to train.log
    env = {**os.environ, "PYTHONUNBUFFERED": "1", **(extra_env or {})}
    print("Running:", " ".join(cmd), flush=True)
    last_bar = 0.0
    with open(log_path, "a") as log:
        log.write(f"\n===== {' '.join(cmd)} =====\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, env=env)
        for line in proc.stdout:
            is_bar = "it/s]" in line or "s/it]" in line or "%|" in line
            if is_bar and "100%|" not in line:
                now = time.time()
                if now - last_bar < 30:
                    continue
                last_bar = now
            print(line.rstrip(), flush=True)
            log.write(line.rstrip() + "\n"); log.flush()
        proc.wait()
    return proc.returncode

def is_done(mode):
    # both train_cached.py and train.py (stage1) write <mode>_history.json at the very end
    return (MODELS_ROOT / mode / f"{mode}_history.json").exists()

def build_cmd(mode):
    hp = HPARAMS[mode]
    common = ["--mode", mode,
              "--data-dir", str(KAGGLE_DATASET_MOUNT),
              "--output-dir", str(MODELS_ROOT / mode),
              "--epochs", str(hp["epochs"]),
              "--batch-size", str(hp["batch_size"]),
              "--lr", str(hp["lr"])]
    if mode == "stage1":
        return ([sys.executable, "-u", str(project_root / "src" / "train.py")] + common +
                ["--data-source", "local", "--num-workers", str(NUM_WORKERS)]), {}
    script = project_root / "src" / "train_cached.py"
    return ([sys.executable, "-u", str(script)] + common +
            ["--feature-cache", str(FEATURE_CACHE_DIR)]), {
        "BRANCH_PROJECT_ROOT": str(project_root),
        "BRANCH_FEATURE_CACHE": str(FEATURE_CACHE_DIR),
    }

status = {}
for mode in BRANCHES:
    print(f"\n{'=' * 70}\n  BRANCH: {mode}\n{'=' * 70}", flush=True)
    if is_done(mode) and not FORCE_RETRAIN:
        print(f"{mode}: already finished (found {mode}_history.json) -- skipping.")
        status[mode] = "skipped (already done)"
        continue
    cmd, extra_env = build_cmd(mode)
    rc = run_and_log(cmd, extra_env)
    sync_once(min_age=0)   # push this branch's checkpoints right away
    status[mode] = "ok" if rc == 0 else f"FAILED (exit code {rc})"
    print(f"\n{mode}: {status[mode]}", flush=True)

print("\nSummary:")
for mode, s in status.items():
    print(f"  {mode:15s} {s}")



  BRANCH: semantic_only
semantic_only: already finished (found semantic_only_history.json) -- skipping.

  BRANCH: physics_only
physics_only: already finished (found physics_only_history.json) -- skipping.

  BRANCH: prnu_only
prnu_only: already finished (found prnu_only_history.json) -- skipping.

  BRANCH: stage1
Running: /usr/bin/python3 -u /kaggle/working/ai-vs-real-face-detector/ai-vs-real-face-detector/src/train.py --mode stage1 --data-dir /kaggle/input/datasets/sahilsawantspit/ai-vs-real-face-detector-data --output-dir /kaggle/working/models_aug/stage1 --epochs 4 --batch-size 32 --lr 0.0001 --data-source local --num-workers 3
ACCELERATOR INFORMATION
Device: cuda
GPU: Tesla T4
CUDA: 12.8


TRAINING CONFIGURATION
Mode: stage1
Data: /kaggle/input/datasets/sahilsawantspit/ai-vs-real-face-detector-data
Data source: local
Output: /kaggle/working/models_aug/stage1
Backbone: efficientnet_b0
Epochs: 4
Batch size: 32
Learning rate: 0.0001
Validation ratio: 0.15
Workers: 3
Checkpoint inte

No files have been modified since last commit. Skipping to prevent empty commit.


[enumeration] processed 80,000 files across 6 directories; found 79,999 images
[enumeration] processed 90,000 files across 6 directories; found 89,999 images
[enumeration] processed 100,000 files across 7 directories; found 99,999 images


No files have been modified since last commit. Skipping to prevent empty commit.


synced meta.json
[enumeration] processed 110,000 files across 8 directories; found 109,999 images
[enumeration] processed 120,000 files across 8 directories; found 119,999 images
[enumeration] processed 130,000 files across 9 directories; found 129,999 images
[enumeration] processed 140,000 files across 9 directories; found 139,999 images
[enumeration] processed 150,000 files across 10 directories; found 149,999 images
[enumeration] processed 160,000 files across 10 directories; found 159,999 images
[enumeration] processed 170,000 files across 11 directories; found 169,999 images
[enumeration] processed 180,000 files across 11 directories; found 179,999 images
[enumeration] processed 190,000 files across 12 directories; found 189,999 images
[enumeration] processed 200,000 files across 13 directories; found 199,999 images
[enumeration] processed 210,000 files across 13 directories; found 209,999 images
[enumeration] processed 220,000 files across 15 directories; found 219,999 images
[en

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

synced stage1_best.pt


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.




STAGE 1 COMPLETE
Best validation accuracy: 0.9952238805970149
Held-out test: {
  "loss": 2.8528031003588983,
  "acc": 0.6397512788851649
}
Checkpoint: /kaggle/working/models_aug/stage1/stage1_best.pt
synced stage1_history.json

stage1: ok

Summary:
  semantic_only   skipped (already done)
  physics_only    skipped (already done)
  prnu_only       skipped (already done)
  stage1          ok


In [19]:
# 6. Compare the branches, then final sync
stop_sync.set()

def fmt(x):
    return "   -  " if x is None else f"{x:.4f}"

comparison = {}
print(f"{'branch':15s} {'best val acc':>13s} {'test acc':>10s} {'test AUC':>10s} {'test F1':>10s}")
for mode in BRANCHES:
    hist = MODELS_ROOT / mode / f"{mode}_history.json"
    if not hist.exists():
        print(f"{mode:15s} {'(not finished)':>13s}")
        continue
    h = json.loads(hist.read_text())
    t = h.get("test", {})
    acc = t.get("accuracy", t.get("acc"))
    auc = t.get("roc_auc")
    f1 = t.get("f1_score", t.get("f1"))
    print(f"{mode:15s} {fmt(h.get('best_val_acc')):>13s} {fmt(acc):>10s} {fmt(auc):>10s} {fmt(f1):>10s}")
    comparison[mode] = {
        "best_val_acc": h.get("best_val_acc"),
        "test": {k: v for k, v in t.items() if k != "by_source"},
        "test_by_source": t.get("by_source"),
    }

comparison_path = Path("/kaggle/working/branch_comparison.json")
comparison_path.write_text(json.dumps(comparison, indent=2))

sync_once(min_age=0)
api.upload_file(path_or_fileobj=str(comparison_path), path_in_repo="models_v2/branch_comparison.json",
                repo_id=HF_CHECKPOINT_REPO, repo_type="model")
if log_path.exists():
    api.upload_file(path_or_fileobj=str(log_path), path_in_repo="logs/train_branches.log",
                    repo_id=HF_CHECKPOINT_REPO, repo_type="model")
print("\nDone:", f"https://huggingface.co/{HF_CHECKPOINT_REPO}")


branch           best val acc   test acc   test AUC    test F1
semantic_only          0.9450     0.6724     0.7942     0.5466
physics_only           0.7500     0.6288     0.6539     0.6234
prnu_only              0.6288     0.5516     0.5596     0.5017
stage1                 0.9952     0.6398        -          -  

Done: https://huggingface.co/sahilSpit/checkpoints


# Notes

- Use **"Save & Run All (Commit)"** for long runs (`stage1`); the three cached branches are short
  enough to run interactively. Commit mode shows no live output, so watch `logs/train_branches.log`
  on the HF repo (synced every 2 min) or run interactively to see progress.
- Checkpoints land on HF as `models/<mode>/<mode>_best.pt` (best validation accuracy) and
  `models/<mode>/<mode>_last.pt`; the per-branch comparison is `models/branch_comparison.json`.
- Pull one onto another machine with:
  ```python
  from huggingface_hub import hf_hub_download
  path = hf_hub_download(repo_id="sahilSpit/checkpoints",
                         filename="models/prnu_only/prnu_only_best.pt")
  ```
- To retrain a single branch, set `BRANCHES = ["prnu_only"]` and `FORCE_RETRAIN = True`.
